# 03 - Validacao e Carga COFIEX

**Objetivo:** ler a base tratada gerada por `limpeza_transformacao.ipynb`, aplicar regras de qualidade, criar flags de repeticao e cronologia, gerar o relatorio de qualidade por coluna e regravar a base tratada com as flags finais.

Fluxo:
1. `limpeza_transformacao.ipynb` gera `dados_tratados/cofiex_tratado.csv`.
2. **Este notebook** le esse CSV, valida, adiciona flags e gera `dados_tratados/relatorio_qualidade_colunas.csv`.

Regra (AGENTS.md): nenhum registro removido por inconsistencia. Tudo vira flag e relatorio.

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd

## Ler a base tratada

Le o CSV gerado pela etapa de limpeza/transformacao. As colunas de data sao convertidas de novo (CSV nao guarda dtype datetime).

In [26]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_TRATADOS = BASE_DIR / "dados_tratados"
CAMINHO_TRATADO = DIR_TRATADOS / "cofiex_tratado.csv"

df_tratado = pd.read_csv(CAMINHO_TRATADO, encoding="utf-8-sig")

colunas_data = [
    "dt_ultimo_recebimento", "dt_primeira_cofiex", "dt_validade_recomendacao",
    "dt_reuniao_negociacao", "dt_publicacao_dou", "dt_assinatura",
    "dt_ultimo_desembolso_original", "dt_ultimo_desembolso_vigente",
]
for c in colunas_data:
    df_tratado[c] = pd.to_datetime(df_tratado[c], errors="coerce")

print("base tratada lida:", df_tratado.shape)

base tratada lida: (3548, 67)


## Validar se valores financeiros negativos existem

In [27]:
neg_fin = (df_tratado["vl_financiamento_dolar"] < 0)
neg_contra = (df_tratado["vl_contrapartida_dolar"] < 0)
print("financiamento negativo:", neg_fin.sum())
print("contrapartida negativa:", neg_contra.sum())
df_tratado["flag_valor_negativo"] = neg_fin | neg_contra
print("registros com valor negativo:", df_tratado["flag_valor_negativo"].sum())

financiamento negativo: 0
contrapartida negativa: 0
registros com valor negativo: 0


## Validar valores financeiros zerados

In [28]:
df_tratado["flag_financiamento_zero"] = (df_tratado["vl_financiamento_dolar"] == 0)
df_tratado["flag_contrapartida_zero"] = (df_tratado["vl_contrapartida_dolar"] == 0)
print("financiamento zero:", df_tratado["flag_financiamento_zero"].sum())
print("contrapartida zero:", df_tratado["flag_contrapartida_zero"].sum())

financiamento zero: 29
contrapartida zero: 254


## Validar datas fora de ordem (cronologia geral)

Sequencia esperada: `dt_ultimo_recebimento` <= `dt_primeira_cofiex` <= `dt_assinatura` <= `dt_ultimo_desembolso_vigente`. Cada par so comparado quando ambas datas validas.

In [29]:
pares = [
    ("dt_ultimo_recebimento", "dt_primeira_cofiex"),
    ("dt_primeira_cofiex", "dt_assinatura"),
    ("dt_assinatura", "dt_ultimo_desembolso_vigente"),
    ("dt_reuniao_negociacao", "dt_assinatura"),
    ("dt_ultimo_desembolso_original", "dt_ultimo_desembolso_vigente"),
]
for ant, post in pares:
    m = df_tratado[ant].notna() & df_tratado[post].notna() & (df_tratado[ant] > df_tratado[post])
    print(f"{ant} > {post}: {m.sum()}")

dt_ultimo_recebimento > dt_primeira_cofiex: 75
dt_primeira_cofiex > dt_assinatura: 37
dt_assinatura > dt_ultimo_desembolso_vigente: 1
dt_reuniao_negociacao > dt_assinatura: 4
dt_ultimo_desembolso_original > dt_ultimo_desembolso_vigente: 8


## Validar recebimento posterior a primeira COFIEX

In [30]:
flag_receb_pos_cofiex = (
    df_tratado["dt_ultimo_recebimento"].notna()
    & df_tratado["dt_primeira_cofiex"].notna()
    & (df_tratado["dt_ultimo_recebimento"] > df_tratado["dt_primeira_cofiex"])
)
print("recebimento posterior a primeira COFIEX:", flag_receb_pos_cofiex.sum())

recebimento posterior a primeira COFIEX: 75


## Validar negociacao posterior a assinatura

In [31]:
flag_negoc_pos_assin = (
    df_tratado["dt_reuniao_negociacao"].notna()
    & df_tratado["dt_assinatura"].notna()
    & (df_tratado["dt_reuniao_negociacao"] > df_tratado["dt_assinatura"])
)
print("negociacao posterior a assinatura:", flag_negoc_pos_assin.sum())

negociacao posterior a assinatura: 4


## Validar assinatura posterior ao desembolso

In [32]:
flag_assin_pos_desemb = (
    df_tratado["dt_assinatura"].notna()
    & df_tratado["dt_ultimo_desembolso_vigente"].notna()
    & (df_tratado["dt_assinatura"] > df_tratado["dt_ultimo_desembolso_vigente"])
)
print("assinatura posterior ao desembolso:", flag_assin_pos_desemb.sum())

assinatura posterior ao desembolso: 1


## Validar desembolso vigente anterior ao original

In [33]:
# Validar desembolso vigente anterior ao original
flag_desemb_vig_ant_orig = (
    df_tratado["dt_ultimo_desembolso_original"].notna()
    & df_tratado["dt_ultimo_desembolso_vigente"].notna()
    & (df_tratado["dt_ultimo_desembolso_vigente"] < df_tratado["dt_ultimo_desembolso_original"])
)
print("desembolso vigente anterior ao original:", flag_desemb_vig_ant_orig.sum())

desembolso vigente anterior ao original: 8


## Consolidar flag_cronologia_invalida

In [34]:
flag_desemb_vig_ant_orig = (
    df_tratado["dt_ultimo_desembolso_original"].notna()
    & df_tratado["dt_ultimo_desembolso_vigente"].notna()
    & (df_tratado["dt_ultimo_desembolso_vigente"] < df_tratado["dt_ultimo_desembolso_original"])
)
df_tratado["flag_cronologia_invalida"] = (
    flag_receb_pos_cofiex | flag_negoc_pos_assin | flag_assin_pos_desemb | flag_desemb_vig_ant_orig
)
print("desembolso vigente anterior ao original:", flag_desemb_vig_ant_orig.sum())
print("TOTAL cronologia invalida:", df_tratado["flag_cronologia_invalida"].sum())

desembolso vigente anterior ao original: 8
TOTAL cronologia invalida: 88


## Flags de repeticao: cd_pleito e nu_operacao

`cd_pleito` nao e chave unica. Marca registros repetidos sem remover (AGENTS.md).

In [35]:
df_tratado["flag_cd_pleito_repetido"] = df_tratado["cd_pleito"].duplicated(keep=False)
df_tratado["qtd_ocorrencias_cd_pleito"] = (
    df_tratado.groupby("cd_pleito")["cd_pleito"].transform("size")
)

# nu_operacao: so conta repeticao entre valores nao nulos
nu_nao_nulo = df_tratado["nu_operacao"].notna()
df_tratado["flag_nu_operacao_repetida"] = (
    nu_nao_nulo & df_tratado["nu_operacao"].duplicated(keep=False)
)

print("linhas com cd_pleito repetido:", df_tratado["flag_cd_pleito_repetido"].sum())
print("codigos cd_pleito distintos repetidos:",
      df_tratado.loc[df_tratado["flag_cd_pleito_repetido"], "cd_pleito"].nunique())
print("linhas com nu_operacao repetida:", df_tratado["flag_nu_operacao_repetida"].sum())

linhas com cd_pleito repetido: 342
codigos cd_pleito distintos repetidos: 158
linhas com nu_operacao repetida: 82


## Relatorio de qualidade por coluna

Para cada coluna original: tipo, nulos, percentual de nulos, unicos, percentual de unicos.

In [36]:
colunas_originais = [
    "cd_pleito","nu_processo_sei","nm_pleito","sg_pleito","nm_proponente",
    "de_tipo_operacao","de_fase","de_fonte","sg_fonte","nm_moeda",
    "vl_financiamento_dolar","vl_contrapartida_dolar","de_esfera","nm_regiao",
    "nm_uf","sg_uf","dt_ultimo_recebimento","dt_primeira_cofiex",
    "dt_validade_recomendacao","dt_reuniao_negociacao","dt_publicacao_dou",
    "dt_assinatura","dt_ultimo_desembolso_original","dt_ultimo_desembolso_vigente",
    "nu_operacao",
]
n = len(df_tratado)
relatorio = pd.DataFrame({
    "coluna": colunas_originais,
    "tipo": [str(df_tratado[c].dtype) for c in colunas_originais],
    "nulos": [df_tratado[c].isna().sum() for c in colunas_originais],
    "pct_nulos": [round(df_tratado[c].isna().sum()/n*100, 2) for c in colunas_originais],
    "unicos": [df_tratado[c].nunique(dropna=False) for c in colunas_originais],
    "pct_unicos": [round(df_tratado[c].nunique(dropna=False)/n*100, 2) for c in colunas_originais],
})
relatorio = relatorio.sort_values("pct_nulos", ascending=False).reset_index(drop=True)
relatorio

,coluna,tipo,nulos,pct_nulos,unicos,pct_unicos
0,dt_reuniao_negociacao,datetime64[ns],2630,74.13,830,23.39
1,dt_validade_recomendacao,datetime64[ns],2555,72.01,289,8.15
2,dt_publicacao_dou,datetime64[ns],2456,69.22,219,6.17
3,nu_processo_sei,object,2454,69.17,1027,28.95
4,dt_ultimo_desembolso_original,datetime64[ns],2315,65.25,800,22.55
5,nu_operacao,object,2146,60.48,1337,37.68
6,dt_ultimo_desembolso_vigente,datetime64[ns],2139,60.29,893,25.17
7,dt_assinatura,datetime64[ns],2132,60.09,1106,31.17
8,dt_primeira_cofiex,datetime64[ns],1630,45.94,394,11.10
9,vl_contrapartida_dolar,float64,1059,29.85,1262,35.57


##  Comparar linhas antes e depois do tratamento

In [37]:
# Comparar linhas antes e depois do tratamento
qtd_bruto = pd.read_excel(ARQUIVO_BRUTO, sheet_name="Dados").shape[0]
qtd_tratado = len(df_tratado)

print("linhas base bruta:", qtd_bruto)
print("linhas df_tratado:", qtd_tratado)
print("diferenca:", qtd_tratado - qtd_bruto)
assert qtd_tratado == qtd_bruto, "Quantidade de linhas mudou no tratamento!"
print("OK: nenhuma linha removida")

linhas base bruta: 3548
linhas df_tratado: 3548
diferenca: 0
OK: nenhuma linha removida


## Carga: salvar relatorio e regravar base tratada com flags

Gera `relatorio_qualidade_colunas.csv` e regrava `cofiex_tratado.csv` com as flags de validacao. Tudo em `utf-8-sig`.

In [38]:
CAMINHO_TRATADO = DIR_TRATADOS / "cofiex_tratado.csv"
CAMINHO_RELATORIO = DIR_TRATADOS / "relatorio_qualidade_colunas.csv"

relatorio.to_csv(CAMINHO_RELATORIO, index=False, encoding="utf-8-sig")
df_tratado.to_csv(CAMINHO_TRATADO, index=False, encoding="utf-8-sig")

print("relatorio salvo:", CAMINHO_RELATORIO)
print("base tratada regravada:", CAMINHO_TRATADO, df_tratado.shape)

relatorio salvo: C:\Users\heron\Downloads\CofiexBI\dados_tratados\relatorio_qualidade_colunas.csv
base tratada regravada: C:\Users\heron\Downloads\CofiexBI\dados_tratados\cofiex_tratado.csv (3548, 67)


In [39]:
# Teste de releitura
df_teste = pd.read_csv(CAMINHO_TRATADO, encoding="utf-8-sig")
df_rel_teste = pd.read_csv(CAMINHO_RELATORIO, encoding="utf-8-sig")
assert len(df_teste) == len(df_tratado), "Linhas divergem!"
print("LEITURA OK")
print("base:", df_teste.shape, "| relatorio:", df_rel_teste.shape)

LEITURA OK
base: (3548, 67) | relatorio: (25, 6)


## Resumo final das validacoes

In [40]:
resumo_validacao = pd.Series({
    "valor_negativo": df_tratado["flag_valor_negativo"].sum(),
    "financiamento_zero": df_tratado["flag_financiamento_zero"].sum(),
    "contrapartida_zero": df_tratado["flag_contrapartida_zero"].sum(),
    "cronologia_invalida": df_tratado["flag_cronologia_invalida"].sum(),
    "cd_pleito_repetido": df_tratado["flag_cd_pleito_repetido"].sum(),
    "nu_operacao_repetida": df_tratado["flag_nu_operacao_repetida"].sum(),
})
print("linhas preservadas:", len(df_tratado))
resumo_validacao

linhas preservadas: 3548


valor_negativo            0
financiamento_zero       29
contrapartida_zero      254
cronologia_invalida      88
cd_pleito_repetido      342
nu_operacao_repetida     82
dtype: int64